In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

import struct

sns.set_theme()

We will attempt to better classify the LogisticRegression data with K-Nearest Neighbors.

In [4]:
data_dir = "/home/patrick/Projects/DataScienceAndMachineLearning/Data/"

names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num", "idk"]
df = pd.read_table(data_dir + 'cleve.mod', sep = r'\s+', names=names)
# df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
# drop columns that cannot easily be converted to meaningful numbers
df = df.drop(columns = ["idk", "cp", "restecg", "slope", "thal", "fbs", "exang"])
df = df.replace("male", 0.0).replace("fem", 1.0).replace("true", 0.0).replace("fal", 0.0)
df = df.replace("buff", 0.0).replace("sick", 1.0)
df = df.replace("?", np.nan).dropna(axis=0, how="any")
df = df.astype(float)
for col in df:
    max_col = max(df[col])
    if max_col > 1e-15:
        df[col] /= max_col
df

/tmp/ipykernel_13297/1947319172.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("male", 0.0).replace("fem", 1.0).replace("true", 0.0).replace("fal", 0.0)
/tmp/ipykernel_13297/1947319172.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("buff", 0.0).replace("sick", 1.0)


,age,sex,trestbps,chol,thalach,oldpeak,ca,num
0,0.818182,0.0,0.725,0.413121,0.742574,0.370968,0.000000,0.0
1,0.870130,0.0,0.800,0.507092,0.534653,0.241935,1.000000,1.0
2,0.870130,0.0,0.600,0.406028,0.638614,0.419355,0.666667,1.0
3,0.480519,0.0,0.650,0.443262,0.925743,0.564516,0.000000,0.0
4,0.532468,1.0,0.650,0.361702,0.851485,0.225806,0.000000,0.0
...,...,...,...,...,...,...,...,...
298,0.623377,0.0,0.620,0.452128,0.866337,0.000000,0.666667,0.0
299,0.740260,0.0,0.660,0.367021,0.831683,0.000000,0.000000,0.0
300,0.636364,0.0,0.590,0.264184,0.623762,0.129032,1.000000,1.0
301,0.961039,1.0,0.600,0.476950,0.599010,0.032258,0.333333,0.0


In [29]:
# some distance functions
def euclidean_distance(p, q):
    return np.sqrt((p - q) @ (p - q))

def manhattan_distance(p, q):
    return sum(p-q)

In [ ]:
euclidean_distance(df.iloc[0, :], df.iloc[1, :])

np.float64(1.4411920923224375)

In [35]:
train_cutoff = round(0.7 * df.shape[0])
train_X = df.iloc[:train_cutoff, :-1].to_numpy()
test_X = df.iloc[train_cutoff:, :-1].to_numpy()
train_y = df.iloc[:train_cutoff, -1].to_numpy()
test_y = df.iloc[train_cutoff:, -1].to_numpy()
train_X

array([[0.81818182, 0.        , 0.725     , ..., 0.74257426, 0.37096774,
        0.        ],
       [0.87012987, 0.        , 0.8       , ..., 0.53465347, 0.24193548,
        1.        ],
       [0.87012987, 0.        , 0.6       , ..., 0.63861386, 0.41935484,
        0.66666667],
       ...,
       [0.63636364, 1.        , 0.67      , ..., 0.8019802 , 0.        ,
        0.        ],
       [0.54545455, 0.        , 0.6       , ..., 0.8019802 , 0.        ,
        0.        ],
       [0.53246753, 0.        , 0.55      , ..., 0.75742574, 0.        ,
        0.        ]], shape=(209, 7))

In [52]:
def get_k_nearest_neighbors(dist_func, train_X, train_y, point, k):
    # build distance dictionary
    distances = []
    for i in range(np.shape(train_X)[0]):
        dist = dist_func(train_X[i], point)
        distances.append([train_X[i], train_y[i], dist])
    return sorted(distances, key=lambda x: x[-1])[:k]

def predict_k_nearest_neighbors(dist_func, train_x, train_y, point, k):
    nearest_neighbors = get_k_nearest_neighbors(dist_func, train_x, train_y, point, k)
    prediction = 0.0
    for i in range(k):
        neighbor = nearest_neighbors[i]
        neighbor_prediction = neighbor[1]
        prediction += neighbor_prediction / k
    return float(round(prediction))


In [89]:
# ok now test!!!
def test_k_nearest_neighbors(dist_func, train_X, train_y, test_X, test_y, k):
    good_preds = 0
    for i in range(test_X.shape[0]):
        pred = predict_k_nearest_neighbors(dist_func, train_X, train_y, test_X[i, :], k)
        good_preds += 1 if pred == test_y[i] else 0
    good_preds /= test_X.shape[0]
    return good_preds

Table of prediction vs. k count


In [90]:
print("  k | Successful prediction rate on test data with Euclidean distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(euclidean_distance, train_X, train_y, test_X, test_y, k)
    print(f" {k:2d} | {pred_rate:1.3f}")

  k | Successful prediction rate on test data with Euclidean distance
----|----------------------------------------------------------------
  1 | 0.753
  2 | 0.719
  3 | 0.809
  4 | 0.775
  5 | 0.820
  6 | 0.798
  7 | 0.820
  8 | 0.820
  9 | 0.843
 10 | 0.843
 11 | 0.843
 12 | 0.843
 13 | 0.843
 14 | 0.843
 15 | 0.843
 16 | 0.854
 17 | 0.831
 18 | 0.831
 19 | 0.831
 20 | 0.809


In [91]:
print("  k | Successful prediction rate on test data with Manhattan distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(manhattan_distance, train_X, train_y, test_X, test_y, k)
    print(f" {k:2d} | {pred_rate:1.3f}")

  k | Successful prediction rate on test data with Manhattan distance
----|----------------------------------------------------------------
  1 | 0.449
  2 | 0.551
  3 | 0.551
  4 | 0.551
  5 | 0.551
  6 | 0.551
  7 | 0.551
  8 | 0.551
  9 | 0.551
 10 | 0.551
 11 | 0.551
 12 | 0.551
 13 | 0.551
 14 | 0.551
 15 | 0.551
 16 | 0.551
 17 | 0.551
 18 | 0.551
 19 | 0.551
 20 | 0.551


# PCA

In [92]:
# first preprocess data differently
data_dir = "/home/patrick/Projects/DataScienceAndMachineLearning/Data/"

names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num", "idk"]
df = pd.read_table(data_dir + 'cleve.mod', sep = r'\s+', names=names)
# df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
# drop columns that cannot easily be converted to meaningful numbers
df = df.drop(columns = ["idk", "cp", "restecg", "slope", "thal", "fbs", "exang"])
df = df.replace("male", 0.0).replace("fem", 1.0).replace("true", 0.0).replace("fal", 0.0)
df = df.replace("buff", 0.0).replace("sick", 1.0)
df = df.replace("?", np.nan).dropna(axis=0, how="any")
df = df.astype(float)
train_df = df.iloc[:, :-1]
for col in train_df:
    mean = np.mean(train_df[col])
    std = np.std(train_df[col])
    train_df[col] = (train_df[col] - mean) / std
train_df

/tmp/ipykernel_114510/1356487009.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("male", 0.0).replace("fem", 1.0).replace("true", 0.0).replace("fal", 0.0)
/tmp/ipykernel_114510/1356487009.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("buff", 0.0).replace("sick", 1.0)


,age,sex,trestbps,chol,thalach,oldpeak,ca
0,0.941719,-0.689382,0.760757,-0.268426,0.023254,1.070920,-0.720134
1,1.385406,-0.689382,1.611115,0.754623,-1.807192,0.382574,2.482850
2,1.385406,-0.689382,-0.656507,-0.345637,-0.891969,1.329049,1.415189
3,-1.942248,-0.689382,-0.089602,0.059722,1.635789,2.103438,-0.720134
4,-1.498561,1.450575,-0.089602,-0.828207,0.982058,0.296531,-0.720134
...,...,...,...,...,...,...,...
298,-0.722108,-0.689382,-0.429745,0.156236,1.112804,-0.908073,1.415189
299,0.276188,-0.689382,0.023780,-0.770298,0.807730,-0.908073,-0.720134
300,-0.611186,-0.689382,-0.769888,-1.889861,-1.022715,-0.219728,2.482850
301,2.161858,1.450575,-0.656507,0.426475,-1.240626,-0.735987,0.347527


In [110]:
A = train_df.to_numpy()
U, S, V = np.linalg.svd(A)
k = 2
A_new = pd.DataFrame(A @ V[:, :k])
A_new

,0,1
0,-1.300018,-0.002486
1,1.248673,-1.673070
2,0.407057,0.215006
3,-0.682655,0.903210
4,0.200619,1.769605
...,...,...
293,0.809725,-0.635335
294,-0.846507,-0.249722
295,2.694558,0.202166
296,0.066992,0.294967


apply PCA transformation to y data as well lol

In [111]:
PCA_train_X = A_new.iloc[:train_cutoff, :].to_numpy()
PCA_test_X = A_new.iloc[train_cutoff:, :].to_numpy()
PCA_train_y = df.iloc[:train_cutoff, -1].to_numpy()
PCA_test_y = df.iloc[train_cutoff:, -1].to_numpy()

In [112]:
print("  k | Successful prediction rate on test data with Euclidean distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(euclidean_distance, PCA_train_X, PCA_train_y, PCA_test_X, PCA_test_y, k)
    print(f" {k:2d} | {pred_rate:1.3f}")

  k | Successful prediction rate on test data with Euclidean distance
----|----------------------------------------------------------------
  1 | 0.640
  2 | 0.618
  3 | 0.663
  4 | 0.685
  5 | 0.652
  6 | 0.640
  7 | 0.652
  8 | 0.663
  9 | 0.674
 10 | 0.674
 11 | 0.674
 12 | 0.685
 13 | 0.663
 14 | 0.652
 15 | 0.674
 16 | 0.674
 17 | 0.685
 18 | 0.697
 19 | 0.685
 20 | 0.674


In [114]:
print(" PCA dim | Highest achieved prediction rate on test data with Euclidean distance with k = 1 ... 20")
for dim in range(1, 8):
    A = train_df.to_numpy()
    U, S, V = np.linalg.svd(A)
    A_new = pd.DataFrame(A @ V[:, :dim])
    PCA_train_X = A_new.iloc[:train_cutoff, :].to_numpy()
    PCA_test_X = A_new.iloc[train_cutoff:, :].to_numpy()
    PCA_train_y = df.iloc[:train_cutoff, -1].to_numpy()
    PCA_test_y = df.iloc[train_cutoff:, -1].to_numpy()
    max_pred_rate = -1.0
    for k in range(1, 21):
        pred_rate = test_k_nearest_neighbors(euclidean_distance, PCA_train_X, PCA_train_y, PCA_test_X, PCA_test_y, k)
        if pred_rate > max_pred_rate:
            max_pred_rate = pred_rate

    print(f" {dim:2d} | {max_pred_rate:1.3f}")

 PCA dim | Highest achieved prediction rate on test data with Euclidean distance with k = 1 ... 20
  1 | 0.663
  2 | 0.697
  3 | 0.674
  4 | 0.753
  5 | 0.820
  6 | 0.787
  7 | 0.809
